In [ ]:
# Install required packages
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install transformers numpy scikit-learn matplotlib seaborn Pillow

# Data Augmentation and Mixing

This notebook contains data augmentation code for creating balanced and imbalanced datasets from brain CT scan images.

## Balanced Dataset Augmentation
Creates a 1:1 ratio of Normal and Stroke classes using classic augmentation and mixing techniques.

In [ ]:
# Standard library imports
import os
import random

# Third-party imports
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from PIL import Image

IMAGE_SIZE = (256, 256)

# --- Augmentation pipelines ---

classic_augment = T.Compose([
    T.RandomRotation(degrees=15),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=5),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    T.Normalize(mean=[0.5], std=[0.5])
])

unmix_augment = T.Compose([
    T.RandomAdjustSharpness(sharpness_factor=2, p=0.5),
    T.RandomAutocontrast(p=0.5),
    T.Normalize(mean=[0.5], std=[0.5])
])

# Mixing function
def mix_tensors(t1, t2, alpha=0.5):
    return alpha * t1 + (1 - alpha) * t2

# --- Paths ---
input_folder = "Dataset"
output_folder = "Balanced_dataset"
os.makedirs(output_folder, exist_ok=True)

# --- Dataset ---
resize_and_tensor = T.Compose([
    T.Resize(IMAGE_SIZE),
    T.ToTensor()
])
dataset = ImageFolder(root=input_folder, transform=resize_and_tensor)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

# --- Class specific config ---
augmentation_counts = {
    "Normal": {"classic": 5, "mixed": 4},  # +1 original = 10
    "Stroke": {"classic": 10, "mixed": 9}  # +1 original = 20
}

# --- Build class index map for mixing ---
class_to_indices = {}
for idx, (path, label) in enumerate(dataset.imgs):
    class_name = dataset.classes[label]
    class_to_indices.setdefault(class_name, []).append(idx)

# --- Save function ---
def save_image(tensor, path):
    tensor = tensor * 0.5 + 0.5  # Unnormalize
    pil = T.ToPILImage()(tensor.squeeze(0))
    pil.save(path)

def save_augmented_images(image_tensor, image_name, class_name, output_folder, classic_n, mixed_n):
    class_folder = os.path.join(output_folder, class_name)
    os.makedirs(class_folder, exist_ok=True)

    # Save original
    save_image(image_tensor, os.path.join(class_folder, f"{image_name}_orig.png"))

    # Classic augmentations
    for i in range(classic_n):
        aug_tensor = classic_augment(image_tensor)
        save_image(aug_tensor, os.path.join(class_folder, f"{image_name}_aug_{i}.png"))

    # Mixing augmentations
    indices_pool = class_to_indices[class_name]
    for i in range(mixed_n):
        other_idx = random.choice(indices_pool)
        other_img_path = dataset.imgs[other_idx][0]
        other_img = Image.open(other_img_path).convert("RGB")
        other_img = other_img.resize(IMAGE_SIZE)
        other_tensor = T.ToTensor()(other_img).unsqueeze(0)

        mixed = mix_tensors(image_tensor, other_tensor, alpha=random.uniform(0.4, 0.6))
        mixed = unmix_augment(mixed)
        save_image(mixed, os.path.join(class_folder, f"{image_name}_mix_{i}.png"))

# --- Main loop ---
print("🚀 Starting image augmentation and mixing...")

for idx, (image_tensor, label) in enumerate(dataloader):
    image_path = dataset.imgs[idx][0]
    image_name = os.path.splitext(os.path.basename(image_path))[0]
    class_name = dataset.classes[label.item()]
    aug_config = augmentation_counts.get(class_name, {"classic": 0, "mixed": 0})

    save_augmented_images(
        image_tensor,
        image_name,
        class_name,
        output_folder,
        classic_n=aug_config["classic"],
        mixed_n=aug_config["mixed"]
    )

print("✅ Balanced classic + mixed augmentation done. ~120k images generated.")

## Imbalanced Dataset Augmentation
Creates augmented dataset with MixUp-style mixing for handling imbalanced classes.

In [ ]:
import os
import random
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Classic augmentation pipeline
classic_aug = T.Compose([
    T.RandomRotation(degrees=15),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=5),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    T.Normalize(mean=[0.5], std=[0.5])
])

# Mixing augmentation (MixUp-style)
def mixing_augmentation(img1, img2, alpha=0.4):
    lam = random.uniform(alpha, 1.0)
    return lam * img1 + (1 - lam) * img2

# Paths
input_folder = "Dataset"
output_folder = "Aug_dataset"
os.makedirs(output_folder, exist_ok=True)

resize_shape = (256, 256)

base_transform = T.Compose([
    T.Resize(resize_shape),
    T.ToTensor()
])

classic_aug = T.Compose([
    T.RandomRotation(degrees=15),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=5),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    T.Normalize(mean=[0.5], std=[0.5])
])

dataset = ImageFolder(root=input_folder, transform=base_transform)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

base_image_count = len(dataset)
total_augmented_images = 120000
aug_per_image = total_augmented_images // base_image_count

# Save augmented images
def save_augmented_images(image_tensor, image_name, class_name, output_folder, dataset, current_idx, num_augmentations=20):
    class_folder = os.path.join(output_folder, class_name)
    os.makedirs(class_folder, exist_ok=True)

    for i in range(num_augmentations):
        if i < num_augmentations // 2:
            aug_tensor = classic_aug(image_tensor)
        else:
            rand_idx = random.randint(0, len(dataset) - 1)
            if rand_idx == current_idx:
                rand_idx = (rand_idx + 1) % len(dataset)
            mix_tensor, _ = dataset[rand_idx]
            aug_tensor = mixing_augmentation(image_tensor.squeeze(0), mix_tensor)
            aug_tensor = T.Normalize(mean=[0.5], std=[0.5])(aug_tensor)

        unnormalized_tensor = aug_tensor * 0.5 + 0.5
        if unnormalized_tensor.dim() == 4:
            unnormalized_tensor = unnormalized_tensor.squeeze(0)

        augmented_pil = T.ToPILImage()(unnormalized_tensor)
        aug_filename = f"{image_name}_aug_{i}.png"
        augmented_pil.save(os.path.join(class_folder, aug_filename))


# Process all images
print("🚀 Starting augmentation to generate ~120k images...")

for idx, (image_tensor, label) in enumerate(dataloader):
    image_name = os.path.splitext(os.path.basename(dataset.imgs[idx][0]))[0]
    class_name = dataset.classes[label.item()]
    save_augmented_images(image_tensor, image_name, class_name, output_folder,
                          dataset=dataset, current_idx=idx,
                          num_augmentations=aug_per_image)

print("✅ All augmented images saved.")

## MRI Augmentation Best Practices

### Medical Image Considerations:
When augmenting MRI or brain CT scans, it's crucial to maintain clinical relevance and avoid introducing artifacts that could mislead the model.

### Recommended Augmentation Pipeline for MRI:

#### 1. **Conservative Geometric Transforms**
- **Rotation**: Limited to ±10-15 degrees (brain orientation matters)
- **Translation**: Small shifts (±10% of image size)
- **Scaling**: Minimal scaling (0.95-1.05) to preserve anatomical proportions
- **Flipping**: Horizontal flipping acceptable, vertical flipping should be avoided

#### 2. **Intensity Augmentations**
- **Brightness/Contrast**: Simulate different MRI sequences (T1, T2, FLAIR)
- **Gamma Correction**: Adjust image intensity curves
- **Histogram Equalization**: Enhance contrast while preserving details

#### 3. **Medical-Specific Augmentations**
- **Gaussian Noise**: Model thermal noise in MRI acquisition
- **Motion Blur**: Simulate patient movement artifacts
- **Elastic Deformation**: Model tissue movement (use sparingly)
- **Intensity Inhomogeneity**: Simulate bias field artifacts

#### 4. **Validation Considerations**
- Always validate augmented images with medical experts
- Ensure augmentations don't create false pathological features
- Maintain class-specific anatomical constraints
- Consider using domain-specific augmentation libraries

### Alternative Libraries for Medical Imaging:
- **TorchIO**: Specifically designed for medical image augmentation
- **MONAI**: Medical imaging framework with robust augmentation tools
- **SimpleITK**: Advanced medical image processing capabilities

### Quality Control:
- Visual inspection of augmented samples
- Statistical analysis of intensity distributions
- Validation against original dataset characteristics
- Expert review for clinical relevance

In [ ]:
import os
import cv2
import random
from glob import glob
import albumentations as A

# Define augmentation pipeline
augmentation_pipeline = A.Compose([
    A.Rotate(limit=15, p=0.5),
    A.HorizontalFlip(p=0.5),
    A.RandomResizedCrop(scale=(0.8, 1.0), ratio=(0.9, 1.1), size=(224, 224), p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=0, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.ElasticTransform(alpha=1.0, sigma=50.0, alpha_affine=50.0, p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
    A.CoarseDropout(max_holes=1, max_height=32, max_width=32, min_holes=1, min_height=16, min_width=16, fill_value=0, p=0.3)
])

# Input and output directories
input_base_dir = 'Stroke_classification'  # contains subfolders: Normal, Haemorrhagic, Ischemic
output_base_dir = 'augmented_dataset'

# Class-wise configuration
class_config = {
    'Normal': {'current': 3721, 'target': 40000},
    'Haemorrhagic': {'current': 2298, 'target': 40000},
    'Ischemic': {'current': 422, 'target': 40000}
}

# Create output directories
for cls in class_config:
    os.makedirs(os.path.join(output_base_dir, cls), exist_ok=True)

# Augmentation function
def augment_and_save(class_name, input_dir, output_dir, current_count, target_count):
    image_paths = glob(os.path.join(input_dir, '*.jpg')) + glob(os.path.join(input_dir, '*.png')) + glob(os.path.join(input_dir, '*.jpeg'))
    num_to_generate = target_count - current_count
    counter = 0

    while counter < num_to_generate:
        img_path = random.choice(image_paths)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        augmented = augmentation_pipeline(image=img)['image']
        augmented = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)
        save_path = os.path.join(output_dir, f"{class_name}_aug_{counter}.jpg")
        cv2.imwrite(save_path, augmented)
        counter += 1

# Run augmentation for each class
for cls, cfg in class_config.items():
    print(f"Augmenting class: {cls}")
    augment_and_save(
        class_name=cls,
        input_dir=os.path.join(input_base_dir, cls),
        output_dir=os.path.join(output_base_dir, cls),
        current_count=cfg['current'],
        target_count=cfg['target']
    )

print("Augmentation complete. Augmented images saved in 'augmented_dataset' folder.")